# 01 Data Setup and Embeddings
500 reviewed human proteins are fetched from UniProt, ESM2 embeddings are computed, the embedding space is explored through EDA, and nearest neighbour search is implemented across three distance metrics.

## Data Fetching and Embedding Computation
The UniProt REST API is queried for reviewed human proteins (organism: Homo sapiens) with GO terms, protein family, and sequence. Each sequence is passed through ESM2 to produce a 320-dimensional embedding vector.

In [ ]:
import pathlib, sys, os

project_root = pathlib.Path.cwd()
for _ in range(5):
    if (project_root / "environment.yml").exists():
        break
    project_root = project_root.parent
sys.path.insert(0, str(project_root))
os.chdir(project_root)

from src.data_loader import fetch_uniprot, compute_embeddings, save_data, load_data
import esm

df = fetch_uniprot(500)
model, alphabet = esm.pretrained.esm2_t6_8M_UR50D()
model.eval()
embeddings = compute_embeddings(df, model, alphabet)
save_data(df, embeddings)
print(embeddings.shape)

## Loading Saved Data

In [ ]:
df2, emb2 = load_data()
print(df2.shape)    
print(emb2.shape)   

## Exploratory Data Analysis

In [ ]:
print("Missing GO terms:", df2["go_terms"].isna().sum())
print("Missing sequences:", df2["sequence"].isna().sum())
print("Missing family labels:", df2["family"].isna().sum())

print("\nUnique families:", df2["family"].nunique())
print("\nTop 10 families:\n", df2["family"].value_counts().head(10))
print("\nSequence length stats:\n", df2['sequence'].str.len().describe())

In [ ]:
df_labeled = df2[df2["family"].notna()].reset_index(drop=True)
emb_labeled = emb2[df2["family"].notna()]
print(f"Proteins with family labels: {len(df_labeled)}")

### Embedding Quality Check
The distribution of L2 norms is inspected across all embeddings. A tight distribution indicates cosine and Euclidean distances will behave similarly.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

l2_norms = np.linalg.norm(emb2, axis=1)

plt.figure(figsize=(8, 4))
plt.hist(l2_norms, bins=20, edgecolor='black')
plt.xlabel('L2 Norm')
plt.ylabel('Count')
plt.title('Histogram of Embedding L2 Norms')
plt.tight_layout()
plt.show()

## Nearest neighbours

In [ ]:
print(df2[df2["uniprot_id"] == "P01308"])

In [ ]:
from src.search import nearest_neighbours

insulin_idx = df2[df2["uniprot_id"] == "P01308"].index[0]
insulin_emb = emb2[insulin_idx]

print("Cosine")
print(nearest_neighbours(insulin_emb, emb2, df2, metric="cosine", n=5))

print("Euclidean")
print(nearest_neighbours(insulin_emb, emb2, df2, metric="euclidean", n=5))

print("Manhattan")
print(nearest_neighbours(insulin_emb, emb2, df2, metric="manhattan", n=5))

## Summary
- 500 reviewed human proteins fetched with 340 unique protein families
- 117 proteins missing family labels — retained for unsupervised analysis
- L2 norms tightly distributed (~4.5–6.0), making cosine and Euclidean distances broadly equivalent
- All three distance metrics (cosine, Euclidean, Manhattan) return identical top-5 neighbours for insulin, suggesting the embedding geometry is robust to metric choice